# STFT and CWT Image Generation for CWRU Segments

This notebook converts every 1-second vibration segment from the CWRU dataset into two types of 2-D time–frequency images: **STFT spectrograms** and **CWT scalograms**. These images form the complete input dataset used later for CNN training and for all XAI analyses.

**1. Inputs and outputs**  
We load the segmented dataset (`data_segments.npz`) produced in the previous notebook.  
All generated images are saved automatically under:

Images_STFT_CWT/  
 ├── STFT/  
 │  └── {train, test}/{Baseline, Inner, Outer, Ball}/  
 ├── CWT/  
 │  └── {train, test}/{Baseline, Inner, Outer, Ball}/  

Each image corresponds to one vibration segment and follows a consistent naming convention.

**2. STFT image generation**  
For every segment we compute:
- Hann-window STFT (20 ms window, 75% overlap)  
- magnitude in dB  
- frequency range 50–6000 Hz  
- dynamic clipping to highlight structure  
- resizing to 256×256  
- conversion to RGB using the jet colormap  

**3. CWT image generation**  
We compute Morlet-based scalograms using:
- wavelet `cmor1.5-1.0` (MATLAB-like “amor”)  
- frequencies 50–6000 Hz  
- magnitude in dB  
- resize to 256×256  
- RGB conversion  

**4. Train/test reconstruction**  
The notebook automatically identifies whether the NPZ file uses unified arrays (`X_all`) or separated arrays (`X_train`, `X_test`) and assigns each segment to its correct split.

The output is a fully standardized, reproducible 256×256 STFT and CWT image dataset for all 2930 CWRU segments, ready for CNN modeling and interpretability workflows.


In [1]:
import numpy as np
from pathlib import Path
from skimage.transform import resize
import imageio.v2 as imageio
import matplotlib.cm as cm
from scipy.signal import spectrogram
import pywt

# ----------------------------------------------------------------------
# Paths and constants (relative to notebook location)
# ----------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()

# Organized data folder created in Notebook 01
ROOT = PROJECT_ROOT / "data_CWRU_Organized"

# Segments NPZ created in the data-preparation notebook
SEG_NPZ = ROOT / "data_segments.npz"

# Where we will save all STFT/CWT images
OUT_ROOT = PROJECT_ROOT / "Images_STFT_CWT"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

FS_GLOBAL = 12000  # 12 kHz, as in Chen2020

CLASS_MAP = {
    0: "Baseline",
    1: "Inner",
    2: "Outer",
    3: "Ball",
}

TARGET_SHAPE = (256, 256)  # (freq, time) for both STFT and CWT

# ----------------------------------------------------------------------
# STFT: image (non-plotting, mirrors plot_spectrogram)
# ----------------------------------------------------------------------
def stft_image(sig, fs=FS_GLOBAL,
               win_sec=0.02, overlap=0.75,
               fmin=50, fmax=6000,
               dyn_range=20,
               target_shape=TARGET_SHAPE):
    """
    Compute STFT (Hann) and return a (H,W) float image in dB,
    cropped to [fmin, fmax] and resized to target_shape.
    """
    nperseg = int(win_sec * fs)
    if nperseg < 64:
        nperseg = 64
    noverlap = int(nperseg * overlap)

    f, t, Sxx = spectrogram(sig, fs=fs, window="hann",
                            nperseg=nperseg, noverlap=noverlap,
                            mode="magnitude")

    Sxx_db = 20 * np.log10(Sxx + 1e-12)

    # Focus on top dyn_range dB
    vmax = np.max(Sxx_db)
    vmin = vmax - dyn_range
    Sxx_db = np.clip(Sxx_db, vmin, vmax)

    # Select frequency band
    mask = (f >= fmin) & (f <= fmax)
    img = Sxx_db[mask, :]

    # Resize to target shape
    if target_shape is not None:
        H, W = target_shape
        img = resize(img, (H, W), preserve_range=True, anti_aliasing=True)

    return img.astype(np.float32)


# ----------------------------------------------------------------------
# CWT: image (non-plotting, mirrors plot_cwt)
# ----------------------------------------------------------------------
def cwt_image(sig, fs=FS_GLOBAL,
              f_min=50, f_max=6000, n_freqs=224,
              dyn_range=60,
              target_shape=TARGET_SHAPE):
    """
    Compute Morlet-based CWT scalogram (MATLAB-like) and return
    a (H,W) float image in dB, resized to target_shape.
    """
    wavelet = "cmor1.5-1.0"        # complex Morlet ~ 'amor'
    fc = pywt.central_frequency(wavelet)

    freqs = np.linspace(f_min, f_max, n_freqs)
    scales = fc * fs / freqs

    coef, _ = pywt.cwt(sig, scales, wavelet, sampling_period=1/fs)
    mag = np.abs(coef)
    mag_db = 20 * np.log10(mag + 1e-12)

    vmax = np.max(mag_db)
    vmin = vmax - dyn_range
    img = np.clip(mag_db, vmin, vmax)

    if target_shape is not None:
        H, W = target_shape
        img = resize(img, (H, W), preserve_range=True, anti_aliasing=True)

    return img.astype(np.float32)


# ----------------------------------------------------------------------
# Helper: normalize to 0–255 and save PNG
# ----------------------------------------------------------------------
def save_img_png(img_float, out_path, cmap_name="jet"):
    """
    Convert a 2D float image to RGB using a matplotlib colormap
    and save as PNG.
    """
    # Normalize 0–1
    img = img_float - img_float.min()
    if img.max() > 0:
        img = img / img.max()

    # Apply colormap → RGBA → RGB
    cmap = cm.get_cmap(cmap_name)
    rgba = cmap(img)                     # (H, W, 4)
    rgb = (rgba[..., :3] * 255).astype(np.uint8)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    imageio.imwrite(out_path, rgb)


# ----------------------------------------------------------------------
# Load segments NPZ and figure out layout (all vs split)
# ----------------------------------------------------------------------
data = np.load(SEG_NPZ, allow_pickle=True)
print("NPZ keys:", list(data.keys()))

if "X_all" in data:
    # unified arrays + index splits
    X_all       = data["X_all"]
    y_class_all = data["y_class_all"]
    y_load_all  = data["y_load_all"]
    train_idx   = data["train_idx"]
    test_idx    = data["test_idx"]

    N = X_all.shape[0]
    print(f"Using X_all with {N} segments")

    # Build a split label array for convenience
    split_all = np.array(["train"] * N, dtype=object)
    split_all[test_idx] = "test"

    def get_segment(i):
        return X_all[i], int(y_class_all[i]), int(y_load_all[i]), split_all[i], i

else:
    # Fallback: separate train/test arrays
    X_train       = data["X_train"]
    y_class_train = data["y_class_train"]
    y_load_train  = data["y_load_train"]
    X_test        = data["X_test"]
    y_class_test  = data["y_class_test"]
    y_load_test   = data["y_load_test"]

    N_train = X_train.shape[0]
    N_test  = X_test.shape[0]
    print(f"Using split arrays: {N_train} train, {N_test} test")

    def get_segment(i):
        # 0..N_train-1 -> train; N_train..N_train+N_test-1 -> test
        if i < N_train:
            seg   = X_train[i]
            cls   = int(y_class_train[i])
            load  = int(y_load_train[i])
            split = "train"
            idx   = i
        else:
            j     = i - N_train
            seg   = X_test[j]
            cls   = int(y_class_test[j])
            load  = int(y_load_test[j])
            split = "test"
            idx   = i  # global index
        return seg, cls, load, split, idx

    N = N_train + N_test


# ----------------------------------------------------------------------
# Main loop: generate STFT & CWT PNGs
# ----------------------------------------------------------------------
stft_root = OUT_ROOT / "STFT"
cwt_root  = OUT_ROOT / "CWT"

print(f"Generating images into:\n  {stft_root}\n  {cwt_root}")
print(f"Total segments: {N}")

for i in range(N):
    seg, cls_id, load_id, split, global_idx = get_segment(i)
    class_name = CLASS_MAP.get(cls_id, f"Class{cls_id}")

    # --- STFT image ---
    stft_img = stft_image(seg, fs=FS_GLOBAL)
    stft_path = stft_root / split / class_name / f"seg{global_idx:05d}_cls{cls_id}_load{load_id}.png"
    save_img_png(stft_img, stft_path)

    # --- CWT image ---
    cwt_img = cwt_image(seg, fs=FS_GLOBAL)
    cwt_path = cwt_root / split / class_name / f"seg{global_idx:05d}_cls{cls_id}_load{load_id}.png"
    save_img_png(cwt_img, cwt_path)

    if (i + 1) % 100 == 0 or i == N - 1:
        print(f"{i+1}/{N} segments processed", end="\r")


NPZ keys: ['X_train', 'y_class_train', 'y_load_train', 'file_idx_train', 'X_test', 'y_class_test', 'y_load_test', 'file_idx_test', 'file_names', 'perm']
Using split arrays: 2051 train, 879 test
Generating images into:
  C:\Users\Nicolas\XAI\COE691_XAI_Project-Copy1\Images_STFT_CWT\STFT
  C:\Users\Nicolas\XAI\COE691_XAI_Project-Copy1\Images_STFT_CWT\CWT
Total segments: 2930


C:\Users\Nicolas\AppData\Local\Temp\ipykernel_14888\3269316356.py:121: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap(cmap_name)


2930/2930 segments processed